In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'xgboost'

In [2]:
#Load the Dataset
df = pd.read_csv("../data/processed/ltv_dataset.csv")
df.head()

#Check dataset shape:

df.shape

#Check columns and datatypes:

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customerID       7032 non-null   str    
 1   gender           7032 non-null   str    
 2   SeniorCitizen    7032 non-null   int64  
 3   Partner          7032 non-null   str    
 4   Dependents       7032 non-null   str    
 5   tenure           7032 non-null   int64  
 6   PhoneService     7032 non-null   str    
 7   InternetService  7032 non-null   str    
 8   Contract         7032 non-null   str    
 9   MonthlyCharges   7032 non-null   float64
 10  TotalCharges     7032 non-null   float64
 11  Churn            7032 non-null   str    
 12  LTV              7032 non-null   float64
dtypes: float64(3), int64(2), str(8)
memory usage: 714.3 KB


In [3]:
#Define Features and Target

#Your target variable is still:

target = "LTV"

#Create X and y:

X = df.drop(columns=["LTV"])
y = df["LTV"]

#Remove customerID because it is just an identifier:

X = X.drop(columns=["customerID"])

In [4]:
#Define Numerical and Categorical Features

#Use the same feature split as previous days so all models are trained consistently.

#Numerical features
numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SeniorCitizen"
]
#Categorical features
categorical_features = [
    col for col in X.columns if col not in numerical_features
]

#Check them:

print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

Numerical Features: ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
Categorical Features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'InternetService', 'Contract', 'Churn']


In [6]:
#Create Train-Test Split

#Use the same split as earlier models so you can compare results fairly.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

#Check the shapes:

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(5625, 11)
(1407, 11)
(5625,)
(1407,)


In [7]:
#Create Preprocessing Pipelines
#Numeric pipeline
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)
#Categorical pipeline
categorical_transformer = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)
#Combine using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [8]:
#10: Create the XGBoost Pipeline

#Now build the full pipeline with preprocessing + XGBoost.

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=5,
            random_state=42,
            objective="reg:squarederror"
        ))
    ]
)

NameError: name 'XGBRegressor' is not defined

In [9]:
#Train the XGBoost Regressor
xgb_model.fit(X_train, y_train)

NameError: name 'xgb_model' is not defined

In [10]:
#Make Predictions
#Predict on training set

y_train_pred = xgb_model.predict(X_train)

#Predict on test set
y_test_pred = xgb_model.predict(X_test)

#Preview predictions:

y_test_pred[:10]

NameError: name 'xgb_model' is not defined

In [11]:
#Evaluate the XGBoost Model

#Now compute the standard regression metrics.

#MAE
mae = mean_absolute_error(y_test, y_test_pred)
print("MAE:", mae)
#MSE
mse = mean_squared_error(y_test, y_test_pred)
print("MSE:", mse)

#RMSE
rmse = np.sqrt(mse)
print("RMSE:", rmse)
#R² Score
r2 = r2_score(y_test, y_test_pred)
print("R2 Score:", r2)

NameError: name 'y_test_pred' is not defined

In [12]:
#Create a Metrics Summary Table
xgb_metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2 Score"],
    "Value": [mae, mse, rmse, r2]
})

xgb_metrics

NameError: name 'mae' is not defined

In [13]:
#Compare Actual vs Predicted Values
xgb_results = pd.DataFrame({
    "Actual_LTV": y_test.values,
    "Predicted_LTV": y_test_pred
})

xgb_results.head(10)

NameError: name 'y_test_pred' is not defined

In [14]:
#Plot Actual vs Predicted
plt.figure(figsize=(8,6))
sns.scatterplot(x=y_test, y=y_test_pred)
plt.xlabel("Actual LTV")
plt.ylabel("Predicted LTV")
plt.title("Actual vs Predicted LTV - XGBoost")
plt.show()

NameError: name 'plt' is not defined

In [15]:
#Plot Residual Errors

#Residuals = Actual − Predicted

residuals = y_test - y_test_pred

plt.figure(figsize=(8,6))
sns.histplot(residuals, bins=30, kde=True)
plt.title("Residual Error Distribution - XGBoost")
plt.xlabel("Residual Error")
plt.show()

NameError: name 'y_test_pred' is not defined

In [16]:
#Compare All Three Models

comparison_df = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "XGBoost"],
    "MAE": [120.5, 95.2, mae],      # replace with your actual values
    "MSE": [45000.0, 30000.0, mse], # replace with actual values
    "RMSE": [212.1, 173.2, rmse],   # replace with actual values
    "R2 Score": [0.72, 0.81, r2]    # replace with actual values
})

comparison_df

NameError: name 'mae' is not defined

In [17]:
#Save the XGBoost Model

#Save the trained model to the models/ folder:

joblib.dump(xgb_model, "../models/xgboost_ltv.pkl")

#Test loading it:

loaded_xgb = joblib.load("../models/xgboost_ltv.pkl")
loaded_xgb

NameError: name 'xgb_model' is not defined

In [ ]:
## Day 10 Conclusion

An XGBoost Regressor was trained for Customer Lifetime Value prediction.

Key observations:
- XGBoost is a powerful gradient boosting model for tabular data.
- It can capture complex non-linear relationships better than Linear Regression.
- It often performs competitively or better than Random Forest on structured business datasets.

Next step:
- Compare Linear Regression, Random Forest, and XGBoost to select the best LTV model.